# Run the Automated RAG Ingestion Pipeline

Use this notebook to run the Lab 04 ingestion pipeline step by step. It follows `ingest-pipeline.py`, but separates setup, file discovery, index creation, extraction, embedding, and indexing so each part is easier to inspect while you work through the lab.

## 1. Import Packages, Load Configuration, and Choose Run Mode

This cell imports the SDK packages, locates `workshop/.env`, validates required Lab 04 settings, defines shared constants, and sets the notebook run mode.

Set `run_mode` near the bottom of the next cell before running the ingestion step:

- `once`: process new or changed files once
- `reset`: clear `processed_files.json`, then process all files
- `watch`: poll the `data/` folder for new files

`watch_iterations` is only used when `run_mode = "watch"`. It controls how many polling cycles the notebook runs. The original script watches until you press Ctrl+C; in a notebook, a bounded number of cycles is easier to stop and reason about.

In [13]:
import glob
import hashlib
import json
import os
import time
from datetime import datetime
from pathlib import Path

from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    HnswAlgorithmConfiguration,
    SearchField,
    SearchFieldDataType,
    SearchableField,
    SearchIndex,
    SimpleField,
    VectorSearch,
    VectorSearchProfile,
)
from dotenv import load_dotenv
from openai import AzureOpenAI


def find_workshop_dir(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        workshop_dir = candidate / "workshop"
        if (workshop_dir / ".env").exists():
            return workshop_dir
        if candidate.name == "workshop" and (candidate / ".env").exists():
            return candidate
    raise FileNotFoundError("Could not find workshop/.env. Create it from workshop/.env.sample first.")


def required_setting(name: str) -> str:
    value = (os.getenv(name) or "").strip()
    if not value or value.startswith("<"):
        raise ValueError(f"Set {name} in workshop/.env")
    return value


workshop_dir = find_workshop_dir()
lab_dir = workshop_dir / "lab-04-rag-pipeline"
env_path = workshop_dir / ".env"
data_folder = lab_dir / "data"
manifest_file = lab_dir / "processed_files.json"

load_dotenv(env_path, override=True)

foundry_endpoint = required_setting("FOUNDRY_ENDPOINT")
foundry_key = required_setting("FOUNDRY_KEY")
embedding_deployment = required_setting("AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME")
search_endpoint = required_setting("AZURE_SEARCH_ENDPOINT")
search_key = required_setting("AZURE_SEARCH_KEY")

index_name = "rag-content-index"
analyzer_id = "rag_document_analyzer"
poll_interval = 30

# Notebook run controls. Change these before running the ingestion mode cell.
run_mode = "watch"  # Options: "once", "reset", "watch"
watch_iterations = 3  # Used only for run_mode = "watch"; each iteration waits poll_interval seconds.

print(f"Workshop directory: {workshop_dir}")
print(f"Environment file: {env_path}")
print(f"Data folder: {data_folder}")
print(f"Foundry endpoint: {foundry_endpoint}")
print(f"Embedding deployment: {embedding_deployment}")
print(f"Azure AI Search endpoint: {search_endpoint}")
print(f"Search index: {index_name}")
print(f"Analyzer ID: {analyzer_id}")
print(f"Run mode: {run_mode}")
if run_mode == "watch":
    print(f"Watch iterations: {watch_iterations}")

Workshop directory: C:\Users\algut\repos\alesaez\content-processing-solution\workshop
Environment file: C:\Users\algut\repos\alesaez\content-processing-solution\workshop\.env
Data folder: C:\Users\algut\repos\alesaez\content-processing-solution\workshop\lab-04-rag-pipeline\data
Foundry endpoint: https://4ublv6yuwkfh4-aifoundry.services.ai.azure.com/
Embedding deployment: text-embedding-3-large-996913
Azure AI Search endpoint: https://4ublv6yuwkfh4-search.search.windows.net
Search index: rag-content-index
Analyzer ID: rag_document_analyzer
Run mode: watch
Watch iterations: 3


## 2. Track Processed Files and Discover Pending Documents

The manifest stores a SHA-256 hash for each processed file so the pipeline only ingests new or changed files on later runs.

In [14]:
def log(message: str) -> None:
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] {message}")


def load_manifest() -> dict[str, str]:
    if manifest_file.exists():
        with manifest_file.open("r", encoding="utf-8") as file:
            return json.load(file)
    return {}


def save_manifest(manifest: dict[str, str]) -> None:
    with manifest_file.open("w", encoding="utf-8") as file:
        json.dump(manifest, file, indent=2)


def file_hash(file_path: str | Path) -> str:
    sha = hashlib.sha256()
    with Path(file_path).open("rb") as file:
        for block in iter(lambda: file.read(8192), b""):
            sha.update(block)
    return sha.hexdigest()


supported_patterns = ["*.pdf", "*.png", "*.jpg", "*.docx", "*.txt"]


def get_pending_files(manifest: dict[str, str]) -> list[str]:
    all_files: list[str] = []
    for pattern in supported_patterns:
        all_files.extend(glob.glob(str(data_folder / pattern)))

    pending: list[str] = []
    for file_path in sorted(all_files):
        current_hash = file_hash(file_path)
        if manifest.get(file_path) != current_hash:
            pending.append(file_path)
    return pending


manifest = load_manifest()
pending_files = get_pending_files(manifest)

print(f"Files tracked in manifest: {len(manifest)}")
print(f"Pending files: {len(pending_files)}")
for file_path in pending_files:
    print(f"- {Path(file_path).name}")

Files tracked in manifest: 7
Pending files: 0


## 3. Define the Search Index and Chunking Helpers

The index stores extracted text chunks, source file names, generated summary and topics, and a 3072-dimension vector field for embeddings from `text-embedding-3-large`.

In [15]:
def ensure_index(search_endpoint: str, search_key: str) -> None:
    index_client = SearchIndexClient(
        endpoint=search_endpoint,
        credential=AzureKeyCredential(search_key),
    )

    fields = [
        SimpleField(name="id", type=SearchFieldDataType.String, key=True, filterable=True),
        SearchableField(name="content", type=SearchFieldDataType.String),
        SimpleField(name="file_name", type=SearchFieldDataType.String, filterable=True),
        SearchableField(name="summary", type=SearchFieldDataType.String),
        SearchableField(name="key_topics", type=SearchFieldDataType.String),
        SearchField(
            name="content_vector",
            type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
            searchable=True,
            vector_search_dimensions=3072,
            vector_search_profile_name="vector-profile",
        ),
    ]

    vector_search = VectorSearch(
        algorithms=[HnswAlgorithmConfiguration(name="hnsw-algorithm")],
        profiles=[
            VectorSearchProfile(
                name="vector-profile",
                algorithm_configuration_name="hnsw-algorithm",
            )
        ],
    )

    index = SearchIndex(name=index_name, fields=fields, vector_search=vector_search)
    index_client.create_or_update_index(index)


def chunk_content(text: str, max_chars: int = 2000) -> list[str]:
    paragraphs = text.split("\n\n")
    chunks: list[str] = []
    current = ""

    for paragraph in paragraphs:
        paragraph = paragraph.strip()
        if not paragraph:
            continue
        if len(current) + len(paragraph) + 2 > max_chars and current:
            chunks.append(current.strip())
            current = paragraph
        else:
            current += ("\n\n" + paragraph) if current else paragraph

    if current.strip():
        chunks.append(current.strip())

    return chunks if chunks else [text[:max_chars]]


def make_doc_id(file_name: str, chunk_index: int) -> str:
    raw = f"{file_name}::{chunk_index}"
    return hashlib.md5(raw.encode()).hexdigest()


sample_chunks = chunk_content("Paragraph one.\n\nParagraph two.")
print(f"Index and chunking helpers are ready. Sample chunk count: {len(sample_chunks)}")

Index and chunking helpers are ready. Sample chunk count: 1


## 4. Create SDK Clients and Ensure the Search Index

Run this after the analyzer has been created with `create-analyzer.py` or `create-analyzer.ipynb`. This cell creates clients and creates or updates the Azure AI Search index.

In [16]:
cu_client = ContentUnderstandingClient(
    endpoint=foundry_endpoint,
    credential=AzureKeyCredential(foundry_key),
)

openai_client = AzureOpenAI(
    azure_endpoint=foundry_endpoint,
    api_key=foundry_key,
    api_version="2024-06-01",
)

search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=index_name,
    credential=AzureKeyCredential(search_key),
)

log("Verifying search index...")
ensure_index(search_endpoint, search_key)
log(f"Search index '{index_name}' is ready.")

[23:23:00] Verifying search index...
[23:23:02] Search index 'rag-content-index' is ready.


## 5. Define File Ingestion Logic

This function extracts one file with Content Understanding, chunks the markdown content, embeds each chunk, and uploads the chunk documents to Azure AI Search.

In [17]:
def ingest_file(
    file_path: str | Path,
    cu_client: ContentUnderstandingClient,
    openai_client: AzureOpenAI,
    search_client: SearchClient,
    embedding_deployment: str,
) -> int:
    file_path = Path(file_path)
    file_name = file_path.name

    with file_path.open("rb") as file:
        file_data = file.read()

    poller = cu_client.begin_analyze_binary(
        analyzer_id=analyzer_id,
        binary_input=file_data,
    )
    result = poller.result()

    doc_content = ""
    fields: dict[str, object] = {}
    for item in result.contents:
        if hasattr(item, "markdown") and item.markdown:
            doc_content += item.markdown + "\n"
        if hasattr(item, "fields") and item.fields:
            for name, data in item.fields.items():
                if hasattr(data, "value_array") and data.value_array is not None:
                    fields[name] = [value.get("value", str(value)) for value in data.value_array]
                elif hasattr(data, "value"):
                    fields[name] = data.value

    if not doc_content.strip():
        log("    Skipped because no extractable content was found.")
        return 0

    chunks = chunk_content(doc_content)
    summary = fields.get("Summary", "")
    key_topics = (
        ", ".join(fields.get("KeyTopics", []))
        if isinstance(fields.get("KeyTopics"), list)
        else str(fields.get("KeyTopics", ""))
    )

    documents_to_upload = []
    for chunk_index, chunk in enumerate(chunks):
        log(f"    Embedding chunk {chunk_index + 1}/{len(chunks)}...")
        response = openai_client.embeddings.create(
            input=chunk,
            model=embedding_deployment,
        )
        embedding = response.data[0].embedding

        documents_to_upload.append({
            "id": make_doc_id(file_name, chunk_index),
            "content": chunk,
            "file_name": file_name,
            "summary": str(summary) if not isinstance(summary, str) else summary,
            "key_topics": key_topics,
            "content_vector": embedding,
        })

    if documents_to_upload:
        upload_result = search_client.upload_documents(documents=documents_to_upload)
        succeeded = sum(1 for result in upload_result if result.succeeded)
        log(f"    Indexed {succeeded} chunk(s) from {file_name}.")
        return succeeded

    return 0

print("File ingestion helper is ready.")

File ingestion helper is ready.


## 6. Run the Selected Pipeline Mode

This cell uses the `run_mode` and `watch_iterations` values from Cell 3.

The original Python script supports three modes:

- `once`: process new or changed files once, like `python ingest-pipeline.py`
- `reset`: clear `processed_files.json`, then process all files, like `python ingest-pipeline.py --reset`
- `watch`: poll the `data/` folder for new files, like `python ingest-pipeline.py --watch`

For notebook watch mode, `watch_iterations` limits how many polling cycles run. Each cycle waits 30 seconds by default.

In [18]:
def run_ingestion_once(manifest: dict[str, str]) -> bool:
    pending = get_pending_files(manifest)
    if not pending:
        return False

    log(f"Detected {len(pending)} new/updated file(s).")
    total_chunks = 0

    for file_path in pending:
        log(f"  Processing: {Path(file_path).name}")
        chunks = ingest_file(
            file_path,
            cu_client,
            openai_client,
            search_client,
            embedding_deployment,
        )
        total_chunks += chunks

        manifest[file_path] = file_hash(file_path)
        save_manifest(manifest)

    log(f"Ingestion complete - {len(pending)} file(s), {total_chunks} chunk(s) indexed.")
    return True


def run_pipeline(run_mode: str = "once", watch_iterations: int = 0) -> None:
    mode = run_mode.strip().lower()
    if mode not in {"once", "reset", "watch"}:
        raise ValueError("run_mode must be one of: once, reset, watch")

    if mode == "reset":
        if manifest_file.exists():
            manifest_file.unlink()
            log("Cleared processed-file tracking. All files will be reprocessed.")
        else:
            log("No processed-file tracking exists yet. All discovered files are already pending.")

    manifest = load_manifest()

    if mode in {"once", "reset"}:
        found = run_ingestion_once(manifest)
        if not found:
            log("No new files to ingest - all documents are up to date.")
        return

    log(f"Watching '{data_folder}/' for new documents.")
    log("In notebook mode, polling is bounded by watch_iterations.")

    if watch_iterations <= 0:
        raise ValueError("Set watch_iterations to a positive number when run_mode is 'watch'.")

    for iteration in range(watch_iterations):
        log(f"Watch cycle {iteration + 1}/{watch_iterations}")
        found = run_ingestion_once(manifest)
        if not found:
            log("No new files. Waiting...")
        if iteration < watch_iterations - 1:
            time.sleep(poll_interval)

    log("Watch-mode polling finished.")


run_pipeline(run_mode=run_mode, watch_iterations=watch_iterations)

[23:23:02] Watching 'C:\Users\algut\repos\alesaez\content-processing-solution\workshop\lab-04-rag-pipeline\data/' for new documents.
[23:23:02] In notebook mode, polling is bounded by watch_iterations.
[23:23:02] Watch cycle 1/3
[23:23:02] No new files. Waiting...
[23:23:32] Watch cycle 2/3
[23:23:32] No new files. Waiting...
[23:24:02] Watch cycle 3/3
[23:24:02] No new files. Waiting...
[23:24:02] Watch-mode polling finished.


## 8. Next Step

After ingestion completes, return to the Lab 04 README and run the RAG agent:

```powershell
python workshop/lab-04-rag-pipeline/rag-agent.py
```